# __House Price Estimation Using Gradient Boosting__

#### Wyatt Flatt

## Introduction and Exploratory Analysis


The Ames housing dataset is a collection of observations on houses from Ames, Iowa. This data is used to test advanced machine-learning algorithms' ability to predict the sales prices of houses using a collection of explanatory features by allowing individuals to submit their models’ predictions and receive a score. The explanatory features associated with each observation include a house’s physical characteristics, room information, location in relation to other objects, and condition. This collection of explanatory features includes both quantitative and qualitative information. The observations are already separated into train and test sets, leaving the focus of this project to be maximizing predictive accuracy on the target feature, sales price, using a gradient boosting model.

## Data Exploration

In [ ]:
# Loading packages
import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

In [ ]:
# Loading in data files
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Holding id variable for submission use later on
id = test.Id

# Dropping id since it does not provide any use for modeling
train = train.drop("Id", axis = 1)
test = test.drop("Id", axis = 1)
train

In [ ]:
# Determining the number of explanatory variables and observations in the training data
train.drop("SalePrice", axis = 1).shape

The training data set contains 1,460 observations and 79 explanatory variables.

In [ ]:
# Getting the summary statistics of the training data
train.describe()

In [ ]:
# Determining variable types and number of missing values
train.info();

In [ ]:
# Getting the number of numeric features
train.drop("SalePrice", axis=1).select_dtypes(include=['number']).shape

In [ ]:
# Getting the number of categorical features
train.drop("SalePrice", axis=1).select_dtypes(include=['object']).shape

In [ ]:
# Checking whcih variables have NAs
train.columns[train.isna().any()].tolist()

Only 19 of the 79 features contain missing values within the training data.

In [ ]:
# Getting average sales price of those with and without features

# Average salesprice of houses with and without fences
print(train["SalePrice"][train["Fence"].isna()].mean())
print(train["SalePrice"][~train["Fence"].isna()].mean())

In [ ]:
# Average salesprice of houses with and without a masonry veneer
print(train["SalePrice"][train["MasVnrType"].isna()].mean())
print(train["SalePrice"][~train["MasVnrType"].isna()].mean())

In [ ]:
# Average salesprice of houses with and without alley access
print(train["SalePrice"][train["Alley"].isna()].mean())
print(train["SalePrice"][~train["Alley"].isna()].mean())

In the training data, houses with fencing, a masonry veneer, or alley access have an average sales price that is tens of thousands of dollars different than houses without the given feature. This information is used later for feature engineering.

In [ ]:
# Gathering quantitative explanatory variables
quantitative_vars = ["LotFrontage","LotArea","OverallQual","OverallCond","YearBuilt", # OverallQual and OverallCond are ordinal vars
                "YearRemodAdd","MasVnrArea","BsmtFinSF1","BsmtFinSF2","BsmtUnfSF",
                "TotalBsmtSF","1stFlrSF","2ndFlrSF","LowQualFinSF","GrLivArea",
                "BsmtFullBath","BsmtHalfBath","FullBath","HalfBath","BedroomAbvGr",
                "KitchenAbvGr","TotRmsAbvGrd","Fireplaces","GarageYrBlt","GarageCars",
                "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
                "ScreenPorch","PoolArea","MiscVal","YrSold"]

# Gathering categorical explanatory variables
cat_vars = [x for x in train.columns if x not in quantitative_vars and x != "SalePrice"]


In [ ]:
### Graphing distribution of sales price

# Creating graph settings
fig, ax = plt.subplots();

# Graphing histogram
sn.histplot(x = train["SalePrice"]);
ax.set_title("Distribution of Sales Price");
ax.set_xlabel("Sales Price");

In [ ]:
### Graphing distribution of log of sales price

# Creating graph settings
fig, ax = plt.subplots();

# Graphing histogram
sn.histplot(x = np.log(train["SalePrice"]));
ax.set_title("Distribution of Log of Sales Price");
ax.set_xlabel("Sales Price");

In the training data, the distribution of housing sales prices is right-skewed with large sales price outliers. With a log transformation on sales price, the distribution becomes relatively normal.

In [ ]:
### Graphing SalePrice correlation with each quantitative variable

# Creating a list of quantitative and response variables
quant_response = quantitative_vars + ["SalePrice"]

# Compute correlations for each combination and narrow down to only response correlations
corr = train[quant_response].corr()["SalePrice"].drop("SalePrice").sort_values()

# Creating graph
fig, ax = plt.subplots(figsize = (10,8));

# Correlation bar chart
sn.barplot(x = corr.index, y = corr);
ax.set_xticklabels(ax.get_xticklabels(), rotation=90);
ax.set_title("Quantitative Feature Correlation with Sales Price");
ax.set_xlabel("Feature");
ax.set_ylabel("Correlation with Sales Price");
plt.tight_layout();

As expected, variables related to the quality and size of the house and its garage are most positively related to the sales price of the house. Based on these correlations, we would expect these variables to be some of the most predictive of sales price during the modeling process.

In [ ]:
### Graphing SalePrice vs Overall Quality 

# Creating graph settings
fig, ax = plt.subplots();

# Graphing boxplot
sn.boxplot(x = train["OverallQual"], y = train["SalePrice"]);
ax.set_title("Sales Price vs. Overall House Quality");
ax.set_ylabel("Sales Price");
ax.set_xlabel("Overall House Quality (1-10)");
plt.tight_layout();

From the graph of sales price versus overall quality, we see the average sales price increase as the quality increases, making it apparent that the two are related. Because of the stable and increasing trend of sales price with the increase in the overall quality of the house, the overall quality will be an important variable in modeling.

In [ ]:
### Graphing SalePrice vs Above Ground Living Area 

# Creating graph settings
fig, ax = plt.subplots();

# Graphing scatterpot
sn.scatterplot(x = train["GrLivArea"], y = train["SalePrice"]);
ax.set_title("Sales Price vs. Above Ground Living Area (ft^2)");
ax.set_ylabel("Sales Price");
ax.set_xlabel("Above Ground Living Area (ft^2)");
plt.tight_layout();

Just as with the overall quality of the house, the above-ground living area is highly positively associated with the sales price of the house, with very few houses deviating from this trend.

In [ ]:
### Graphing SalePrice vs Overall Condition

# Creating graph settings
fig, ax = plt.subplots();

# Graphing boxplot
sn.boxplot(x = train["OverallCond"], y = train["SalePrice"]);
ax.set_title("Sales Price vs. Overall House Condition (1-10)");
ax.set_ylabel("Sales Price");
ax.set_xlabel("Overall House Condition (1-10)");
plt.tight_layout();

From the graph of sales price versus overall condition of the house, we see an unexpected relationship. Before investigation, it would make sense to assume that the sales price of a house would tend to increase as the overall condition of the house increased; however, we see this assumed trend fail near an overall condition of 5, average condition. Here, we see the most expensive houses and lots of variability, resulting in a sales price difference of more than $300,000 between the 25th and 75th percentiles. Beyond a condition rating of average, the average house's sales price remains relatively consistent until a rating of 9, where we see an increase once again.

In [ ]:
### Heatmap of the correlation between quantitative variables

# Creating graph settings
fig, ax = plt.subplots(figsize = (16,12));

# Heatmap of correlation between variables
sn.heatmap(train[quantitative_vars].corr(), cmap="rocket_r");
ax.set_title("Correlation Heatmap of Quantitative Features");
plt.tight_layout();

From the feature correlation heatmap, it is apparent that there are a lot of variables that are correlated with each other. Most of these relationships are straightforward, such as the positive correlation between square footage variables and the number of bedrooms or bathrooms. Additionally, the overall quality of the house is largely related to the size and newness of the house and garage, which we expect to see.

In [ ]:
### Graphing SalePrice vs Neighborhood

# Creating graph settings
fig, ax = plt.subplots(figsize = (10,8));

# Graphing boxplot
sn.boxplot(x = train["Neighborhood"], y = train["SalePrice"]);
ax.set_title("Sales Price vs. Neighborhood");
ax.set_ylabel("Sales Price");
ax.set_xticklabels(ax.get_xticklabels(), rotation = 90);
plt.tight_layout();

From the sales price versus neighborhood graph, we can see a sharp difference between the average sales prices for houses in different neighborhoods. Because the sales prices vary so much between neighborhoods, the neighborhood variable will likely be a strong predictor of sales price.

In [ ]:
### Graphing SalePrice vs Exterior Quality

# Creating graph settings
fig, ax = plt.subplots();

# Setting order for categories
order = ["Fa", "TA", "Gd", "Ex"]

# Graphing boxplot
sn.boxplot(x = train["ExterQual"], y = train["SalePrice"], order=order);
ax.set_title("Sales Price vs. Exterior Quality");
ax.set_ylabel("Sales Price");
ax.set_xticklabels(ax.get_xticklabels(), rotation = 90);
ax.set_xlabel("Exterior Quality Rating");
plt.tight_layout();

Based on the sales price versus exterior quality graph, houses tend to have very different sales prices based on the quality of the exterior material on the houses. With such large differences in prices between groups, we expect the exterior quality to be a useful feature in our model.

## Preprocessing

Feature creation was used to capture differences between houses with and without alley access, a masonry veneer, and fencing while reducing the dimensionality of the data. For each of these original features, a new binary variable was created to indicate the presence or absence of the characteristic described in the original feature. By using these binary indicators and removing the original alley, masonry veneer type, and fence features, the model retained information about whether these features exist while reducing the number of categories and encoded features during model creation, reducing computational expense.

In [ ]:
### Creating variables that retain potentially important information
### while reducing the number of features (Feature Engineering)

# Creating a variable that answers 'has alley access?'
train["AlleyAccess"] = train["Alley"].notna().map({True: "Yes", False: "No"})
test["AlleyAccess"] = test["Alley"].notna().map({True: "Yes", False: "No"})

# Creating a variable that answers 'has masonry veneer?'
train["MasVnrYes"] = train["MasVnrType"].notna().map({True: "Yes", False: "No"})
test["MasVnrYes"] = test["MasVnrType"].notna().map({True: "Yes", False: "No"})

# Creating a variable that answers 'has fence?'
train["FenceYes"] = train["Fence"].notna().map({True: "Yes", False: "No"})
test["FenceYes"] = test["Fence"].notna().map({True: "Yes", False: "No"})


To further reduce dimensionality and improve modeling performance, certain ordinal categorical features were transformed into numeric features with equal spacing between categories. These features included the quality of a house’s exterior material, basement, garage, fireplace, kitchen, and heating, as well as the condition of a house’s exterior material, basement, and garage. Each of these features had an ordinal rating system comprised of the following labels: poor, fair, typical/average, good, and excellent. Because these ordinal categories can also be represented using a numeric rating system, they were changed as such. By converting these features from categorical to numeric, this ensured that the ordinal relationship of the categories was maintained during the underlying decision tree splits of the gradient boosting model, instead of being removed during one-hot encoding. In addition, the reduced number of encoded categories that resulted from this alteration reduced the sparsity of the data.

In [ ]:
# Collecting ordinal categorical variables to change to numeric
cols_to_change = ["ExterQual","ExterCond","BsmtQual","BsmtCond","HeatingQC",
                  "KitchenQual","FireplaceQu","GarageQual","GarageCond"]

# Mapping from category to numeric value
mapping = {"Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5}

# Looping through to change train/test features from categorical to numeric
for col in cols_to_change:
        train[col] = train[col].map(mapping).fillna(0).astype(int)
        test[col] = test[col].map(mapping).fillna(0).astype(int)

For many categorical features in the data, NA values were used to represent the lack of a feature rather than missing information. Because the lack of a feature can be highly predictive, multiple features' NA values were imputed with a new category labeled as “None.” This method was applied to the following features: the exposure of the basement, the rating of the basement’s finished area, a second rating of the basement’s finished area (if there were multiple types), the garage type, and the interior finish of the garage.

In [ ]:
### Changing NA values to none in features in which this is true

# Changing the training category values to None if Na represents None
train.fillna({
    "BsmtExposure": "None",
    "BsmtFinType1": "None",
    "BsmtFinType2": "None",
    "GarageType": "None",
    "GarageFinish": "None"
}, inplace=True)

# Changing the testing category values to None if Na represents None
test.fillna({
    "BsmtExposure": "None",
    "BsmtFinType1": "None",
    "BsmtFinType2": "None",
    "GarageType": "None",
    "GarageFinish": "None"
}, inplace=True)

In total, only five features were dropped. The alley, masonry veneer type, and fence features were dropped since they were already reduced into more compact features in the beginning of preprocessing. The other features that were dropped included a categorical feature containing information about miscellaneous housing items that were not covered by other features, and a feature indicating the quality of the house’s pool. Because these features had very few observations split into multiple categories, they were too sparse and likely not informative for modeling purposes.

In [ ]:
### Dropping variables that have too many missing values to impute in a meaningful way
### Or values that I used to feature engineer another variable

# Variables to drop
drop_vars = ["MiscFeature","Fence","PoolQC","MasVnrType","Alley"]

# Dropping variables
train = train.drop(columns=drop_vars, axis=1)
test = test.drop(columns=drop_vars, axis=1)

# Showing new data dimensions
train.drop(columns="SalePrice", axis = 1).shape

Because different preprocessing steps were required for different features within the data set, this stage of preprocessing was implemented via pipelines. Pipelines are used to string together a series of transformations and model-fitting steps on the data being piped in. This allowed for a reproducible combination of preprocessing and model-creation steps using Scikit-learn. 

The primary reason for using pipelines to model in this way was to ensure that preprocessing steps were correctly applied within each training fold during k-fold cross-validation, preventing data leakage. If imputation steps were handled before cross-validation, each model created in each iteration of cross-validation would learn from the imputation, giving an underestimated test error and potentially resulting an incorrect optimal model being chosen. By creating a pipeline, imputation was handled separately in each fold, preventing data leakage.

First, a pipeline was created to handle the preprocessing steps for numeric features. This pipeline was used to impute the missing values within each numeric feature with the median value of that feature. Another pipeline was created to handle the preprocessing steps for categorical features. This pipeline contains two steps, which are as follows: imputing the missing values within each categorical feature with the most frequently occurring category within that feature and one-hot encoding categorical features to convert each separate category of each feature into a binary variable with a value of 1 or 0 to indicate the presence or absence of that category. This encoding is necessary for many models to interpret the presence of a category. Encoding was handled such that if a category of a feature were observed in the testing data but not in the training data, the test value would simply be encoded as a 0 for all categories encoded during the fitting process, essentially ignoring the testing feature’s category. 

Lastly, a final preprocessing pipeline was created by combining the preprocessing pipelines for both numeric and categorical features using Scikit-learn’s ColumnTransformer function, which enabled different transformations to be applied to specified subsets of features within the dataset. This ensured that numeric and categorical preprocessing steps were applied appropriately.


In [ ]:
### Creating pipelines to handle imputation and encoding (Preprocessing)

# Pipeline to impute numeric variables with median value 
number_pipeline = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = "median"))  # Imputes missing values with median of variable
])

# Pipeline to impute categorical variables with mode and encode
cat_pipeline = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = "most_frequent")), # Imputes missing vals with mode of variable
    ('encoding', OneHotEncoder(handle_unknown = "ignore"))  # Performs one hot encoding to create yes/no variables for categorical vars
])

# Gathering final quantitative and categorical variables after feature engineering
# to ensure imputation is applied to correct variables
quantitative_vars.extend(cols_to_change)
categorical_vars = [x for x in train.columns if x not in quantitative_vars and x != "SalePrice"]

# Creating a preprocessing pipeline that handles both forms of preprocessing
preprocess_pipeline = ColumnTransformer(transformers =[
    ('number_transformer', number_pipeline, quantitative_vars), # Transforms variables from quantitative_vars using numeric pipeline
    ('categroical_transformer', cat_pipeline, categorical_vars) # Transforms variables from categorical_vars using categrocial pipeline
]) 

## Models and Cross Validation

The primary modeling framework used for sales price estimation was gradient boosting (GB) regression. This is an ensemble method that combines many weak decision tree regressors, meaning they have few splits, sequentially by fitting new trees to the residuals of the previous tree. The estimated residuals for observations are scaled via a learning rate to avoid over- or under-correcting predictions when added to the existing ensemble. This process is repeated over a fixed number of trees. The GB framework was chosen over other frameworks, such as random forest (RF), because I wanted to use an ensemble method, and GB models often outperform RF models if overfitting is handled.

Within this framework, both standard gradient boosting (GB) and a stochastic gradient boosting (SGB) variants were considered. The stochastic version attempts to reduce overfitting by introducing increased randomness during the underlying decision tree creation process. This is achieved by fitting each tree using a subsample of the training data, decorrelating the sequential trees, potentially preventing overfitting.

The model selection process was performed using five-fold cross-validation over a set grid of hyperparameters. The final model was refit using the combination of hyperparameters that minimized the cross-validated root mean squared error (RMSE). Because our training data contained 1,460 observations, five-fold cross-validation was chosen so that the number of observations was divisible by the number of folds, resulting in 292 observations per fold. Additionally, seeds were set for the gradient boosting model object and cross-validation indices to ensure reproducibility of cross-validation results. RMSE was used as the model selection metric due to the competition scores being evaluated based on the RMSE between the log of predicted sales prices and the log of observed sales prices. 

The hyperparameter grid used in the cross-validation process included values for the number of trees, learning rate, maximum depth of individual trees, minimum number of samples required in a leaf node, and the subsample proportion of the training data used for each tree. The number of trees controls the number of decision trees used in sequence during model creation. If the number of trees is too small, the sequenced trees may not provide enough correction for the residual errors by the end of the sequence. Conversely, if the number is too large, the sequence may overfit from corrections of the sequenced trees by fitting to noise. 

The learning rate is used in conjunction with the number of trees. Each tree’s predictions are scaled using the learning rate before being added to the overall model to avoid over- or under-emphasizing the fix from each tree. If the learning rate is too large, corrections from sequential trees may overshoot the actual correction that was needed. If the learning rate is too low, the overall model may take enormous amounts of time and trees to correct the sequential predictions.

The maximum depth of individual trees controls the complexity of the underlying decision trees. A large number of splits allows the trees to capture more detailed patterns, reducing bias. In contrast, fewer splits account for only the most influential splits based on the features, making the trees more generalized with a lower variance. The minimum number of samples required in a leaf node serves a similar purpose. The fewer observations required in nodes, the more the underlying decision trees can split to capture detailed effects, reducing bias. With a larger requirement, the remaining nodes average many values to achieve the nodes’ predictions, reducing a tree’s variance.

The proportion of the training data considered during tree creation indicates whether the model is a GB or SGB model. If the proportion is anything less than 1, the underlying trees are created on a subset of the training data, meaning the final model is an SGB model. As the subset gets smaller, trees are fit using fewer of the same observations, further decorrelating sequential trees. This helps control the variance of the gradient boosting model.


In [ ]:
### Creating an instance of my model to store as part of the pipeline

# Creating a gradient boosting regressor with a set seed for reproducibility
gb = GradientBoostingRegressor(random_state=0)

# Creating a gradient boosting pipeline that runs preprocessing and then model fitting
gb_pipeline = Pipeline(steps=[
    ('preprocess', preprocess_pipeline),  # Runs the preprocessing of variables
    ('model', gb)                         # Runs the model creation process
])

In [ ]:
# Splitting the train data into features and response
X_train = train.drop(columns = "SalePrice", axis = 1)
y_train = train.SalePrice

In [ ]:
### Determining the best gradient boosting model using cross validation with an extended parameter grid

# Creating a cross-validation object to ensure shuffling of observations occurs with a set seed
cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Creating a hyperparameter grid to use with GridSearchCv
gbm_params = {
    "model__n_estimators": [300, 600, 1000, 1500],   # Controls number of trees
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1], # Controls rate of correction
    "model__max_depth": [2, 3, 4],                   # Controls number of splits
    "model__min_samples_leaf": [1, 3, 5],            # Controls min number of of samples required to split an internal node
    "model__subsample": [0.6, 0.8, 1.0]              # Controls the amount of data to use for building trees (stochastic gradient boosting if < 1.0)
}

# Finding and fitting the model with the optimal hyperparameters
grid_gbm = GridSearchCV(estimator = gb_pipeline,
                    param_grid = gbm_params,
                    cv = cv,
                    scoring = "neg_root_mean_squared_error" # RMSE used here since it is used for score evaluation in challenge
                    )

# Best model
gbm_mod = grid_gbm.fit(X_train, y_train)

## Model Results

Based on the cross-validation procedure, the RMSE-minimizing model was a stochastic gradient boosting model. This model was fit using 1500 trees, a learning rate of .01, a maximum depth of 4 splits, a minimum number of samples required in a leaf node of 1, and 80 percent of the training observations to create individual decision trees. With this SGB model, the sales prices for the test data were predicted and submitted on the Kaggle competition page, resulting in an RMSE of .12859 between the log of the SGB model’s sales price predictions and the log of the observed sales prices.

Feature importances, the proportion of the total decrease in the mean squared error (MSE) from splitting on a given feature across all trees, were used to determine the relative importance of features on predictive ability. The overall quality of a house was determined to be the most important feature, accounting for 49.88 percent of the reduction in the MSE across the entire SGB model. The above-ground living area feature was also largely important, accounting for 11.84 percent of the reduction. These features were expected to be among the most important given their exceedingly large positive correlation with sales price, but the reductions attributed to these features are likely overestimated due to the high correlation between features.

In [ ]:
gbm_mod.best_params_

In [ ]:
print(f"The CV RMSE value for the stochastic gradient boosting model is {round(-1*gbm_mod.best_score_,2)}.")

In [ ]:
# Collecting predictions for training data
y_train_preds_gb = gbm_mod.predict(X_train)

# Training R^2 value calculation
train_r2 = r2_score(y_train, y_train_preds_gb)

# Training R^2 presentation
print(f"The training R-squared value of the SGB model is {round(train_r2,4)}.")

In [ ]:
# Getting feature importance values
importances = gbm_mod.best_estimator_.named_steps["model"].feature_importances_

# Getting preprocessed feature names that correspond to feature importance values
feat_names = gbm_mod.best_estimator_.named_steps["preprocess"].get_feature_names_out()

# Changing feature names to more easily-read names
for i in range(len(feat_names)):
    delete, new_name = feat_names[i].split("__")
    feat_names[i] = new_name

# Creating a data frame for feature importances and sorting by importance size
feature_importance = pd.DataFrame({
    "Feature": feat_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

# Shortening features to consider down to only the ten most important
graph_feats = feature_importance.iloc[0:10,:]

# Graphing features with feature importance
fig, ax = plt.subplots();

# Barplot for feature importance
sn.barplot(x=graph_feats["Importance"], y=graph_feats["Feature"]);
ax.set_title("Proportion of the Reduction in MSE (Importance) by Feature");


In [ ]:
graph_feats

In [ ]:
### Getting predicted values from the stochastic gradient boosting model

# Collecting test observation predictions
gbm_preds = gbm_mod.predict(test)

# Creating prediction data frame
gbm_submit = pd.DataFrame({"Id": np.array(id),
                               "SalePrice": gbm_preds})

gbm_submit

In [ ]:
# Exporting predictions to a csv for kaggle competition upload
gbm_submit.to_csv('gbm_submit.csv', index=False)

## Sources

https://www.geeksforgeeks.org/machine-learning/gradient-boosting-vs-random-forest/

https://www.geeksforgeeks.org/machine-learning/ml-one-hot-encoding/

https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html
